# Agent Orchestration: Building Your Own Framework

Orchestration frameworks like LangGraph and CrewAI abstract common agent patterns. Understanding what they do under the hood — and building a minimal version — is the best way to make informed choices about when to use them versus rolling your own.

## What Orchestration Frameworks Provide

LangGraph, CrewAI, AutoGen, and similar frameworks provide:
- **State management**: persisting agent state across steps
- **Graph-based control flow**: conditional edges, loops, and branches in agent logic
- **Streaming**: token-by-token output and step-by-step progress
- **Human-in-the-loop**: pausing for approval at designated checkpoints
- **Persistence**: saving and resuming agent runs
- **Observability**: tracing, logging, and replay

The tradeoff: frameworks add abstraction overhead and constrain your architecture to their model. For complex, novel agent designs, you often fight the framework rather than leveraging it.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import json

@dataclass
class NodeResult:
    node_name: str
    output: dict
    next_node: Optional[str]
    error: Optional[str] = None

class AgentGraph:
    def __init__(self):
        self.nodes: dict = {}
        self.edges: dict = {}
        self.entry: Optional[str] = None

    def add_node(self, name: str, fn: Callable):
        self.nodes[name] = fn

    def add_edge(self, from_node: str, to_node: str, condition: Callable = None):
        if from_node not in self.edges:
            self.edges[from_node] = []
        self.edges[from_node].append((to_node, condition))

    def set_entry(self, node_name: str):
        self.entry = node_name

    def run(self, initial_state: dict, max_steps: int = 20) -> dict:
        state = dict(initial_state)
        current = self.entry
        execution_log = []
        for step in range(max_steps):
            if not current or current == '__end__':
                break
            if current not in self.nodes:
                state['error'] = f'Unknown node: {current}'
                break
            try:
                result = self.nodes[current](state)
                state.update(result.get('state_update', {}))
                execution_log.append({'step': step+1, 'node': current,
                                       'output': str(result)[:80]})
                next_node = result.get('next')
                if not next_node and current in self.edges:
                    for edge_to, condition in self.edges[current]:
                        if condition is None or condition(state):
                            next_node = edge_to
                            break
                current = next_node
            except Exception as e:
                state['error'] = str(e)
                break
        state['execution_log'] = execution_log
        return state

# Build a simple research-and-summarize graph
graph = AgentGraph()

def research_node(state):
    query = state.get('task', '')
    return {'state_update': {'research': f'Found relevant data about: {query[:50]}'}, 'next': 'analyze'}

def analyze_node(state):
    research = state.get('research', '')
    needs_more = len(research) < 30
    return {'state_update': {'analysis': f'Analysis of: {research[:40]}'}, 'next': 'recheck' if needs_more else 'summarize'}

def recheck_node(state):
    return {'state_update': {'research': state['research'] + ' (extended)'}, 'next': 'analyze'}

def summarize_node(state):
    return {'state_update': {'summary': f'Summary: {state.get("analysis", "")[:60]}'}, 'next': '__end__'}

graph.add_node('research', research_node)
graph.add_node('analyze', analyze_node)
graph.add_node('recheck', recheck_node)
graph.add_node('summarize', summarize_node)
graph.set_entry('research')

result = graph.run({'task': 'Impact of LLMs on software engineering productivity'})
print('Graph execution:')
for log in result['execution_log']:
    print(f'  Step {log["step"]}: [{log["node"]}]')
print(f'Summary: {result.get("summary", "N/A")}')

## Human-in-the-Loop Checkpoints

For high-stakes agent actions (sending emails, making purchases, modifying production data), the agent should pause and request human approval before proceeding. This is the most important safety mechanism for agentic systems.

Checkpoint design principles:
- **Minimal interruption**: only pause for irreversible or high-impact actions
- **Clear context**: show the human exactly what the agent is about to do and why
- **Timeout handling**: if approval times out, fail safely (do nothing) rather than proceeding
- **Audit trail**: log every checkpoint decision with timestamp and approver

In [ ]:
class HumanInLoopAgent:
    def __init__(self, agent_fn: Callable, approval_fn: Callable):
        self.agent = agent_fn
        self.approval = approval_fn  # returns True/False
        self.checkpoint_log = []

    RISKY_ACTIONS = {'send_email', 'delete_file', 'make_payment', 'push_code', 'post_to_slack'}

    def needs_approval(self, action_name: str) -> bool:
        return action_name in self.RISKY_ACTIONS

    def run_with_checkpoints(self, task: str) -> dict:
        plan = self.agent(task)
        approved_steps = []
        skipped_steps = []
        for step in plan:
            action = step.get('action')
            if self.needs_approval(action):
                approved = self.approval(step)
                self.checkpoint_log.append({
                    'action': action, 'args': step.get('args'), 'approved': approved
                })
                if approved:
                    approved_steps.append(step)
                else:
                    skipped_steps.append(step)
            else:
                approved_steps.append(step)
        return {'approved': approved_steps, 'skipped': skipped_steps,
                'checkpoint_log': self.checkpoint_log}

def mock_planner(task):
    return [
        {'action': 'search_web', 'args': {'query': task}},
        {'action': 'summarize', 'args': {'content': '...'}},
        {'action': 'send_email', 'args': {'to': 'user@example.com', 'subject': 'Research complete'}},
    ]

def auto_approve(step):
    print(f'  [CHECKPOINT] Approve {step["action"]}({step["args"]})? -> auto-approving')
    return True

hil_agent = HumanInLoopAgent(mock_planner, auto_approve)
result = hil_agent.run_with_checkpoints('Research and email summary about LLM trends')
print(f'Approved: {len(result["approved"])} actions')
print(f'Skipped: {len(result["skipped"])} actions')
for c in result['checkpoint_log']:
    print(f'  Checkpoint: {c["action"]} -> approved={c["approved"]}')